# Synthetic Eval Comparison: Concat vs Top-1 kNN

Notebook для завтрашнего сравнения:

1. Выбрать 30 daily validation cases, потом переключить на 365.
2. Сгенерировать и сохранить общие sparse masks.
3. Прогнать concat ensemble и построить rank histogram.
4. Найти один ближайший train neighbor по той же mask и сравнить full-field errors.
5. Оставить placeholder для precomputed predictions из `m2m_var_100`; формат подключим отдельно.

Тяжелые ячейки выключены флагами `RUN_* = False`.

In [ ]:
import csv
import json
import os
import platform
import re
from functools import partial
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm

from synthetic_eval.metrics import ensemble_crps, interval_coverage, rank_histogram, spread_skill, weighted_mean
from synthetic_eval.observations import ObservationConfig, make_observation_mask, make_sparse_observation
from synthetic_eval.runners import ConcatRunner
from synthetic_eval.knn_baseline import masked_condition_mse, physical_mse
from utils import NpyImageDataset, channel_normalize, channel_denormalize

if platform.system() == 'Darwin':
    REPO_DIR = Path('/Users/amir/sciml/diffusion_data_assimilation')
    DATA_ROOT = Path('/Users/amir/sciml/sea_ice_data')
else:
    REPO_DIR = Path(os.environ.get('REPO_DIR', '/home'))
    DATA_ROOT = Path(os.environ.get('DATA_ROOT', '/mnt/sciml/a.sadreev/sea_ice_data'))

os.chdir(REPO_DIR)

VALID_DIR = DATA_ROOT / 'valid'
TRAIN_DIR = DATA_ROOT / 'train'
STATS_JSON = DATA_ROOT / 'train' / 'stats.json'
MASK_PATH = DATA_ROOT / 'mask_padding.npy'
PRECOMPUTED_DIR = Path('/mnt/sciml/data_assimilation/exp_for_art/april_2026/m2m_var_100')

OUTPUT_ROOT = DATA_ROOT / 'synthetic_eval_comparison'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

with open(STATS_JSON) as f:
    stats = json.load(f)
CHANNEL_MEAN = tuple(stats['mean'])
CHANNEL_STD = tuple(stats['std'])
VALID_MASK = np.load(MASK_PATH).astype(np.float32)
IMAGE_SIZE = tuple(VALID_MASK.shape)

DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')

print('repo_dir       =', REPO_DIR)
print('data_root      =', DATA_ROOT)
print('valid_dir      =', VALID_DIR)
print('train_dir      =', TRAIN_DIR)
print('output_root    =', OUTPUT_ROOT)
print('device         =', DEVICE)
print('channel_mean   =', CHANNEL_MEAN)
print('channel_std    =', CHANNEL_STD)
print('image_size     =', IMAGE_SIZE)
print('precomputed    =', PRECOMPUTED_DIR)

## Experiment Settings

Начни с `N_CASES = 30`. Для полного года поменяй на `365`.

In [ ]:
N_CASES = 30          # switch to 365 later
DAILY_STRIDE = 24     # hourly data -> one case per day
ENSEMBLE_SIZE = 15

MASK_TYPE = 'swath'   # swath, random, block
DENSITY = 0.05        # used by random/block; recorded for swath
NOISE_STD = 0.0
N_TRACKS_RANGE = (7, 7)
MASK_SEED = 20260407

CHECKPOINT_NAME = 'ema_best_model.pth'
CONCAT_RUN_DIR = None  # set explicitly if auto-search under checkpoints/** is not desired
NUM_TIMESTEPS = 50
METHOD = 'euler'

RANK_STRIDE = 16
KNN_CHANNEL_WEIGHTS = np.array([1.0, 1.0], dtype=np.float32)

RUN_CONCAT = False
RUN_KNN_TOP1 = False

RUN_TAG = f'daily{N_CASES}_m{ENSEMBLE_SIZE}_{MASK_TYPE}_tracks{N_TRACKS_RANGE[0]}-{N_TRACKS_RANGE[1]}'
OUT_DIR = OUTPUT_ROOT / RUN_TAG
MASK_DIR = OUT_DIR / 'masks'
ARRAY_DIR = OUT_DIR / 'arrays'
PLOT_DIR = OUT_DIR / 'plots'
for d in (OUT_DIR, MASK_DIR, ARRAY_DIR, PLOT_DIR):
    d.mkdir(parents=True, exist_ok=True)

print('out_dir =', OUT_DIR)
print('run_tag =', RUN_TAG)

## Select Daily Validation Cases

Выбираем индексы `0, 24, 48, ...`. Это картинки через 24 часа.

In [ ]:
raw_valid_dataset = NpyImageDataset(str(VALID_DIR), preload=False, mmap_mode='r')
case_indices = np.arange(0, len(raw_valid_dataset), DAILY_STRIDE, dtype=int)[:N_CASES]
selected_cases = [
    {'local_case_index': local_idx, 'case_index': int(idx), 'file': raw_valid_dataset.files[int(idx)]}
    for local_idx, idx in enumerate(case_indices)
]

selected_path = OUT_DIR / 'selected_cases.csv'
with open(selected_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['local_case_index', 'case_index', 'file'])
    writer.writeheader()
    writer.writerows(selected_cases)

print('n_selected =', len(selected_cases))
print('first cases:')
for row in selected_cases[:5]:
    print(row)
print('last cases:')
for row in selected_cases[-5:]:
    print(row)

## Generate Shared Masks

Эти masks используются и concat-моделью, и top-1 kNN baseline. Это важно для честного сравнения.

In [ ]:
mask_stack = []
mask_rows = []
for local_idx, case_idx in enumerate(case_indices):
    seed = MASK_SEED + local_idx
    rng = np.random.default_rng(seed)
    obs_config = ObservationConfig(
        mask_type=MASK_TYPE,
        density=DENSITY,
        noise_std=NOISE_STD,
        seed=seed,
        block_size=16,
        n_tracks_range=N_TRACKS_RANGE,
    )
    mask = make_observation_mask(IMAGE_SIZE, obs_config, VALID_MASK, rng)
    mask_stack.append(mask.astype(np.float32))
    np.save(MASK_DIR / f'mask_{local_idx:04d}_case{int(case_idx):06d}.npy', mask.astype(np.float32))
    mask_rows.append({
        'local_case_index': local_idx,
        'case_index': int(case_idx),
        'file': raw_valid_dataset.files[int(case_idx)],
        'seed': seed,
        'observed_fraction': float((mask * VALID_MASK).sum() / max(1.0, VALID_MASK.sum())),
    })
mask_stack = np.stack(mask_stack, axis=0)
np.save(ARRAY_DIR / 'shared_masks.npy', mask_stack)

with open(OUT_DIR / 'mask_metadata.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(mask_rows[0].keys()))
    writer.writeheader()
    writer.writerows(mask_rows)

print('mask_stack shape =', mask_stack.shape)
print('observed fraction mean =', np.mean([r['observed_fraction'] for r in mask_rows]))

## Concat Ensemble And Rank Histogram

Включи `RUN_CONCAT = True` в settings. Ячейка сохраняет generated ensemble и метрики. Rank histogram считается отдельно по каналам.

In [ ]:
def resolve_concat_run_dir(checkpoint_name=CHECKPOINT_NAME):
    if CONCAT_RUN_DIR is not None:
        return str(CONCAT_RUN_DIR)
    candidates = sorted(REPO_DIR.glob(f'checkpoints/**/{checkpoint_name}'))
    if not candidates:
        raise FileNotFoundError(f'No {checkpoint_name} found under {REPO_DIR}/checkpoints/**')
    return str(candidates[-1].parent)

def compute_channel_metrics(ensemble, truth, valid_mask):
    mean = ensemble.mean(axis=0)
    crps = ensemble_crps(ensemble, truth)
    rows = []
    for ch, name in enumerate(['concentration', 'thickness'][:truth.shape[0]]):
        rmse = np.sqrt(weighted_mean((mean[ch] - truth[ch]) ** 2, valid_mask))
        crps_mean = weighted_mean(crps[ch], valid_mask)
        ss = spread_skill(ensemble[:, ch], truth[ch], valid_mask)
        cover = interval_coverage(ensemble[:, ch], truth[ch], levels=(0.5, 0.8, 0.9, 0.95))
        row = {'variable': name, 'rmse': rmse, 'crps': crps_mean, **ss}
        for level, hits in cover.items():
            row[f'coverage_{level}'] = weighted_mean(hits.astype(np.float64), valid_mask)
        rows.append(row)
    return rows

concat_rows = []
rank_counts = {}

if RUN_CONCAT:
    run_dir = resolve_concat_run_dir()
    print('concat_run_dir =', run_dir)
    runner = ConcatRunner(
        run_dir=run_dir,
        checkpoint_name=CHECKPOINT_NAME,
        channel_mean=CHANNEL_MEAN,
        channel_std=CHANNEL_STD,
        num_timesteps=NUM_TIMESTEPS,
        method=METHOD,
        device=DEVICE,
    )

    for local_idx, case_idx in enumerate(tqdm(case_indices, desc='concat cases')):
        x_true = np.asarray(raw_valid_dataset[int(case_idx)], dtype=np.float32)
        mask = mask_stack[local_idx]
        rng = np.random.default_rng(MASK_SEED + local_idx)
        observed = make_sparse_observation(x_true, mask, NOISE_STD, rng)
        ensemble = runner.run(x_true, mask, observed, ENSEMBLE_SIZE, seed=MASK_SEED + 100000 + local_idx)
        np.save(ARRAY_DIR / f'concat_ensemble_{local_idx:04d}_case{int(case_idx):06d}.npy', ensemble.astype(np.float32))

        for row in compute_channel_metrics(ensemble, x_true, VALID_MASK):
            row.update({'local_case_index': local_idx, 'case_index': int(case_idx), 'file': raw_valid_dataset.files[int(case_idx)]})
            concat_rows.append(row)

        rank_mask = VALID_MASK[::RANK_STRIDE, ::RANK_STRIDE] > 0
        for ch, name in enumerate(['concentration', 'thickness'][:x_true.shape[0]]):
            counts = rank_histogram(
                ensemble[:, ch, ::RANK_STRIDE, ::RANK_STRIDE],
                x_true[ch, ::RANK_STRIDE, ::RANK_STRIDE],
                seed=MASK_SEED + local_idx + ch,
                weights_mask=rank_mask,
            )
            rank_counts[name] = rank_counts.get(name, np.zeros_like(counts)) + counts

    with open(OUT_DIR / 'concat_per_case_metrics.csv', 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=sorted({k for r in concat_rows for k in r}))
        writer.writeheader()
        writer.writerows(concat_rows)
    np.savez_compressed(ARRAY_DIR / 'concat_rank_histograms.npz', **rank_counts)
    print('saved concat metrics:', OUT_DIR / 'concat_per_case_metrics.csv')
else:
    print('RUN_CONCAT is False')

In [ ]:
rank_path = ARRAY_DIR / 'concat_rank_histograms.npz'
if rank_path.exists():
    payload = np.load(rank_path)
    for name in payload.files:
        counts = payload[name].astype(np.float64)
        probs = counts / max(1.0, counts.sum())
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.bar(np.arange(probs.size), probs, color='black')
        ax.set_title(f'Concat rank histogram - {name} - {N_CASES} daily cases')
        ax.set_xlabel('Rank of truth')
        ax.set_ylabel('Probability')
        fig.tight_layout()
        fig.savefig(PLOT_DIR / f'concat_rank_hist_{name}.png', dpi=140)
        plt.show()
else:
    print('No concat rank histogram yet:', rank_path)

## Top-1 Nearest Neighbor Baseline

Ищем один nearest train field для каждого выбранного validation day по той же shared mask. Concat здесь не запускается.

In [ ]:
transform = partial(channel_normalize, channel_mean=CHANNEL_MEAN, channel_std=CHANNEL_STD)
norm_valid_dataset = NpyImageDataset(str(VALID_DIR), transform=transform, preload=False, mmap_mode='r')
norm_train_dataset = NpyImageDataset(str(TRAIN_DIR), transform=transform, preload=False, mmap_mode='r')

knn_rows = []
if RUN_KNN_TOP1:
    for local_idx, case_idx in enumerate(tqdm(case_indices, desc='top1 kNN cases')):
        ref_norm = norm_valid_dataset[int(case_idx)]
        ref_raw = np.asarray(raw_valid_dataset[int(case_idx)], dtype=np.float32)
        ref_file = raw_valid_dataset.files[int(case_idx)]
        mask = mask_stack[local_idx]
        observed_norm = ref_norm * torch.from_numpy(mask).to(dtype=ref_norm.dtype).view(1, *mask.shape)

        best = None
        for neighbor_idx in tqdm(range(len(norm_train_dataset)), desc=f'kNN {local_idx:04d}', leave=False):
            neighbor_file = norm_train_dataset.files[neighbor_idx]
            if neighbor_file == ref_file:
                continue
            candidate_norm = norm_train_dataset[neighbor_idx]
            distance = masked_condition_mse(candidate_norm, observed_norm, mask, KNN_CHANNEL_WEIGHTS)
            if best is None or distance < best[0]:
                best = (distance, neighbor_idx, neighbor_file, candidate_norm)

        if best is None:
            raise RuntimeError(f'No neighbor found for {ref_file}')
        distance, neighbor_idx, neighbor_file, candidate_norm = best
        mse = physical_mse(candidate_norm, ref_norm, CHANNEL_MEAN, CHANNEL_STD, VALID_MASK)
        row = {
            'local_case_index': local_idx,
            'case_index': int(case_idx),
            'reference_file': ref_file,
            'neighbor_index': int(neighbor_idx),
            'neighbor_file': neighbor_file,
            'condition_mse_norm': float(distance),
            'observed_fraction': float((mask * VALID_MASK).sum() / max(1.0, VALID_MASK.sum())),
            **mse,
        }
        knn_rows.append(row)

    with open(OUT_DIR / 'knn_top1_metrics.csv', 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=sorted({k for r in knn_rows for k in r}))
        writer.writeheader()
        writer.writerows(knn_rows)
    print('saved top1 kNN metrics:', OUT_DIR / 'knn_top1_metrics.csv')
else:
    print('RUN_KNN_TOP1 is False')

## Quick Comparison Table

После запуска concat и kNN эта ячейка покажет краткое сравнение mean errors.

In [ ]:
def read_csv_rows(path):
    if not Path(path).exists():
        return []
    with open(path) as f:
        return list(csv.DictReader(f))

concat_csv = OUT_DIR / 'concat_per_case_metrics.csv'
knn_csv = OUT_DIR / 'knn_top1_metrics.csv'
concat_rows_disk = read_csv_rows(concat_csv)
knn_rows_disk = read_csv_rows(knn_csv)

if concat_rows_disk:
    for variable in sorted(set(r['variable'] for r in concat_rows_disk)):
        vals = [float(r['rmse']) for r in concat_rows_disk if r['variable'] == variable]
        print(f'concat mean RMSE {variable}: {np.mean(vals):.6g}')
else:
    print('No concat metrics yet')

if knn_rows_disk:
    print(f"top1 kNN mean MSE all channels: {np.mean([float(r['mse_all_channels']) for r in knn_rows_disk]):.6g}")
    print(f"top1 kNN mean RMSE all channels: {np.sqrt(np.mean([float(r['mse_all_channels']) for r in knn_rows_disk])):.6g}")
    for ch in (0, 1):
        key = f'mse_ch{ch}'
        print(f"top1 kNN mean RMSE ch{ch}: {np.sqrt(np.mean([float(r[key]) for r in knn_rows_disk])):.6g}")
else:
    print('No kNN metrics yet')

## Precomputed Predictions Placeholder

Не проверяем сейчас. Завтра, когда будет известен формат папки `m2m_var_100`, сюда добавим loader и приведем predictions к одному из форматов:

- deterministic: `(T, C, H, W)`
- ensemble: `(T, M, C, H, W)`

После этого можно будет переиспользовать те же `compute_channel_metrics` и rank histogram functions.

In [ ]:
print('precomputed_dir placeholder =', PRECOMPUTED_DIR)
print('Format not inspected yet by request.')